# 3. Comparación de controladores lineal y no lineal para el robot aéreo en 2D


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from Robot2D import RobotAereo2D
from Robot2D import animate_robot_from_data

## Controladores

In [ ]:
class ControladorLineal:
    """
    Implementa un controlador lineal en cascada para el robot en 2D
    """
    def __init__(self, m, Ix, kp_z=1.0, kd_z=1.0,
                 kp_y=1.0, kd_y=1.0, kp_phi=1.0, kd_phi=1.0):
        # Ganancias
        self.kp_z = kp_z; self.kd_z = kd_z
        self.kp_y = kp_y; self.kd_y = kd_y
        self.kp_phi = kp_phi; self.kd_phi = kd_phi
        # Parámetros del robot (masa, inercia)
        self.g = 9.81
        self.Ix = Ix    # Para convertir u2_barra a u2
        self.m = m      # Para convertir u1_barra a u1

    def control_z(self, z_des, z, dz_des, dz, ddz_des):
        """
        Controlador PD de la altitud. Genera u1 (propulsión)
        """
        # PD + feedforward
        vz = ddz_des - self.kd_z*(dz-dz_des) - self.kp_z*(z-z_des) 
        # Señal u1 linealizada
        u1 = self.m * (self.g-vz)
        return u1

    def control_y(self, y_des, y, dy_des, dy, ddy_des=0.0):
        """
        Controlador PD lateral. Genera el ángulo phi comandado (phi_c)
        """
        # PD + feedforward
        vy = ddy_des - self.kd_y*(dy-dy_des) - self.kp_y*(y-y_des)  
        # Señal phi comandada
        phi_c = vy/self.g
        return phi_c

    def control_phi(self, phi_des, phi, dphi, dphi_des=0.0):
        """
        Controlador PD para el giro (phi). Genera momento u2 alrededor del eje x
        """
        # PD + feedforward
        vphi = - self.kd_phi*(dphi-dphi_des) - self.kp_phi*(phi-phi_des) 
        # Señal u2 (momento) linealizada
        u2 = vphi * self.Ix
        return u2


class ControladorNoLineal:
    """
    Implementa un controlador no lineal en cascada para el robot en 2D
    """
    def __init__(self, m, Ix, kp_z=1.0, kd_z=1.0,
                 kp_y=1.0, kd_y=1.0, kp_phi=1.0, kd_phi=1.0):
        # Ganancias
        self.kp_z = kp_z; self.kd_z = kd_z
        self.kp_y = kp_y; self.kd_y = kd_y
        self.kp_phi = kp_phi; self.kd_phi = kd_phi
        # Parámetros del robot (masa, inercia)
        self.g = 9.81
        self.Ix = Ix    # Para convertir u2_barra a u2
        self.m = m      # Para convertir u1_barra a u1

    def control_z(self, z_des, z, dz_des, dz, ddz_des, phi):
        """
        Controlador PD de la altitud. Genera u1 (propulsión)
        """
        # PD + feedforward
        vz = ddz_des - self.kd_z*(dz-dz_des) - self.kp_z*(z-z_des) 
        # Señal u1 no lineal
        u1 = self.m * (self.g-vz) / np.cos(phi)
        return u1

    def control_y(self, y_des, y, dy_des, dy, u1, ddy_des=0.0):
        """
        Controlador PD lateral. Genera el ángulo phi comandado (phi_c)
        """
        # PD + feedforward
        vy = ddy_des - self.kd_y*(dy-dy_des) - self.kp_y*(y-y_des)  
        # Señal phi comandada
        x = self.m * vy / u1
        # Limitar al rango +-1 (para poder aplicar arcoseno)
        x = min(0.99, max(x,-0.99))
        # Señal phi no lineal
        phi_c = np.arcsin(x)

        return phi_c

    def control_phi(self, phi_des, phi, dphi, dphi_des=0.0):
        """
        Controlador PD para el giro (phi). Genera momento u2 alrededor del eje x
        """
        # PD + feedforward
        vphi = - self.kd_phi*(dphi-dphi_des) - self.kp_phi*(phi-phi_des) 
        # Señal u2 (momento) linealizada
        u2 = vphi * self.Ix
        return u2

### Generador de una trayectoria

La trayectoria que se utilizará para probar el controlador es la figura 8 descrita por:
$$
\begin{align}
z & = -a_z \sin{\omega_z t} \\
y & = a_y \cos{\omega_y t}
\end{align}
$$
donde $\omega_y = \omega_z / 2$.

In [ ]:
def trayectoria(duracion, omega_z=1.0, az=1.0, ay=1.0, dt=0.01):
    omega_y = omega_z/2
    t=np.linspace(0.0, duracion, int(duracion/dt))

    # Trayectoria deseada para y
    y   =  ay*np.cos(omega_y*t)
    dy  = -ay*omega_y*np.sin(omega_y*t)
    ddy = -ay*omega_y**2*np.cos(omega_y*t)

    # Trayectoria deseada para z
    z   =  -az*np.sin(omega_z*t)
    dz  =  -az*omega_z*np.cos(omega_z*t)
    ddz = az*omega_z**2*np.sin(omega_z*t)

    return (y, dy, ddy), (z, dz, ddz), t

### Trayectoria deseada

In [ ]:
# Parámetros de la trayectoria
tiempo_total = 500.0
# Generación de la trayectoria (z,y)
yy, zz, t = trayectoria(tiempo_total, omega_z=2.0, dt=0.02)
y_des, dy_des, ddy_des = yy
z_des, dz_des, ddz_des = zz

# Tiempo de simulación
dt = t[1]-t[0]

# Forma de la trayectoria deseada
plt.figure(figsize=(5,3))
plt.plot(y_des, z_des)
plt.title("Trayectoria deseada")
plt.xlabel("y [m]"); plt.ylabel("z [m]");

### Seguimiento de trayectoria

In [ ]:
# Ganancias del controlador
kp_z   = 0.1
kd_z   = 10.0
kp_y   = 0.3
kd_y   = 10.0
kp_phi = 150.0
kd_phi = 50.0

# Parámetros estimados del robot
m = 0.2; Ix = 0.1

# Robot simulado
robot = RobotAereo2D()

# Estado inicial del robot
X0 = np.array([y_des[0], z_des[0], 0.1, #np.arctan2(dy_des[0],dz_des[0]),
              dy_des[0], dz_des[0], 0])
robot.set_state(X0)

# Controlador no lineal
nlin_controller = ControladorNoLineal(m, Ix, kp_z, kd_z,
                                      kp_y, kd_y, kp_phi, kd_phi)

# Controlador lineal
lin_controller = ControladorLineal(m, Ix, kp_z, kd_z,
                                   kp_y, kd_y, kp_phi, kd_phi)

# Factor para controlar más rápido el lazo interno
factor_lazo_interno = 5

**Simulación del controlador no lineal**

In [ ]:
# Estado inicial
robot.set_state(X0)

# Almacenamiento del historial de los estados
Xt_nl = np.zeros((t.size, 6))
Xt_nl[0,:] = robot.X

# Ejecución del vuelo
for i in range(0, t.size-1):

    # Estado actual del robot
    y    = robot.X[0];  dy   = robot.X[3]
    z    = robot.X[1];  dz   = robot.X[4]
    phi  = robot.X[2];  dphi = robot.X[5]

    # Controlador
    u1 = nlin_controller.control_z(z_des[i], z, dz_des[i], dz, ddz_des[i], phi)
    phi_c = nlin_controller.control_y(y_des[i], y, dy_des[i], dy, u1, ddy_des[i])

    # Bucle interno (cascada) para controlar phi
    for _ in range(factor_lazo_interno):
        # Controlador para u2
        u2 = nlin_controller.control_phi(phi_c, phi, dphi, 0.0 )

        # Simulación del movimiento del robot
        robot.set_control(u1, u2)
        drone_state = robot.advance_state(dt/factor_lazo_interno)

    # Generación del historial de las posiciones verticales
    Xt_nl[i+1,:] = drone_state

**Simulación del controlador lineal**

In [ ]:
# Estado inicial
robot.set_state(X0)

# Almacenamiento del historial de los estados
Xt_l = np.zeros((t.size, 6))
Xt_l[0,:] = robot.X

# Ejecución del vuelo
for i in range(0, t.size-1):

    # Estado actual del robot
    y    = robot.X[0];  dy   = robot.X[3]
    z    = robot.X[1];  dz   = robot.X[4]
    phi  = robot.X[2];  dphi = robot.X[5]

    # Controlador
    u1 = lin_controller.control_z(z_des[i], z, dz_des[i], dz, ddz_des[i])
    phi_c = lin_controller.control_y(y_des[i], y, dy_des[i], dy, ddy_des[i])

    # Bucle interno (cascada) para controlar phi
    for _ in range(factor_lazo_interno):
        # Controlador para u2
        u2 = lin_controller.control_phi(phi_c, phi, dphi, 0.0 )

        # Simulación del movimiento del robot
        robot.set_control(u1, u2)
        drone_state = robot.advance_state(dt/factor_lazo_interno)

    # Generación del historial de las posiciones verticales
    Xt_l[i+1,:] = drone_state

### Figuras de los resultados

In [ ]:
plt.figure(figsize=(6,6))

plt.plot(y_des, z_des, linestyle='-', marker='.', color='red')
plt.plot(Xt_l[:,0], Xt_l[:,1], linestyle='-', color='blue', alpha=0.5)
plt.plot(Xt_nl[:,0], Xt_nl[:,1], linestyle='-', color='green', alpha=0.5)
plt.title('Trayectoria de vuelo').set_fontsize(10)
plt.xlabel('$y$ [$m$]').set_fontsize(10)
plt.ylabel('$z$ [$m$]').set_fontsize(10)
plt.xticks(fontsize = 10)
plt.yticks(fontsize = 10)
plt.gca().invert_yaxis()
plt.legend(['Trayectoria planeada', 'Controlador lineal', 'Controlador no lineal'],fontsize = 10)
plt.axis('equal')
plt.show()

### Trayectoria del control no lineal

In [ ]:
plt.figure(figsize=(12,6))
plt.tight_layout()

plt.subplot(2,1,1)
plt.plot(t, Xt_nl[:,1]); plt.plot(t,z_des)
plt.title('Trayectoria en z (altura)')
plt.xlabel('$t$ [s]'); plt.ylabel('$z$ [$m$]')
plt.gca().invert_yaxis()
plt.legend(['Ejecutado', 'Deseado'],fontsize = 10)

plt.subplot(2,1,2)
plt.plot(t, Xt_nl[:,0]); plt.plot(t,y_des)
plt.title('Trayectoria en y (lateral)')
plt.xlabel('$t$ [s]'); plt.ylabel('$y$ [$m$]')
plt.legend(['Ejecutado', 'Deseado'],fontsize = 10)
plt.tight_layout()

### Trayectoria del control linealizado

In [ ]:
plt.figure(figsize=(12,6))
plt.subplot(2,1,1)
plt.plot(t, Xt_l[:,1]); plt.plot(t,z_des)
plt.title('Trayectoria en z (altura)')
plt.xlabel('$t$ [s]'); plt.ylabel('$z$ [$m$]')
plt.gca().invert_yaxis()
plt.legend(['Ejecutado', 'Deseado'],fontsize = 10)

plt.subplot(2,1,2)
plt.plot(t, Xt_l[:,0]); plt.plot(t,y_des)
plt.title('Trayectoria en y (lateral)')
plt.xlabel('$t$ [s]'); plt.ylabel('$y$ [$m$]')
plt.legend(['Ejecutado', 'Deseado'],fontsize = 10)
plt.tight_layout()